# Stage 6: Model Training & Comparative Benchmark
## Industrial Machine Predictive Maintenance

**Objectives:**
1. Establish a zero-intelligence baseline using `DummyClassifier`.
2. Compare candidate algorithms: Logistic Regression, Decision Tree, Random Forest, and Gradient Boosting.
3. Use 5-Fold Stratified Cross-Validation.
4. Prioritize **Fault Recall** alongside Macro F1.
5. Train and persist the primary `RandomForestClassifier` with balanced class weights and metadata.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
from src.config import PROCESSED_DATA_DIR, SENSOR_FEATURES, TARGET_COLUMN, MODEL_FILE, METADATA_FILE
from src.train import benchmark_candidate_models, train_and_save_final_model, load_model, load_metadata

### 1. Load Preprocessed Training Data

In [ ]:
train_df = pd.read_csv(PROCESSED_DATA_DIR / "train.csv")
X_train = train_df[SENSOR_FEATURES]
y_train = train_df[TARGET_COLUMN].astype(int)
print(f"Loaded {len(train_df)} training samples.")
train_df.head()

### 2. 5-Fold Stratified Cross-Validation Benchmark

In [ ]:
results_df = benchmark_candidate_models(X_train, y_train, n_splits=5)
display(results_df)

### 3. Fit & Save Final Random Forest Classifier

In [ ]:
model, metadata = train_and_save_final_model(X_train, y_train)
print("Model training complete! Metadata summary:")
print(f"- Model: {metadata['model_name']} (v{metadata['model_version']})")
print(f"- Training Accuracy: {metadata['training_metrics']['accuracy']*100:.2f}%")
print(f"- Fault Recall: {metadata['training_metrics']['fault_recall']*100:.2f}%")

print("\nFeature Importances:")
for feat, imp in metadata['feature_importances'].items():
    print(f"  {feat:12s}: {imp*100:6.2f}%")